# Notebook 06: Feature Importance (Logistic Regression Coefficients)

Trains Logistic Regression on the full dataset (Whisper small) and prints top AD vs CN terms.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

PROJECT_ROOT = Path(".").resolve()
CSV_PATH = PROJECT_ROOT / "outputs" / "features" / "all_texts_small.csv"

df = pd.read_csv(CSV_PATH)
texts = df["text"].values
y = df["label"].map({"cn": 0, "ad": 1}).values

tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    max_features=3000,
    ngram_range=(1,2),
    min_df=2,
)
X = tfidf.fit_transform(texts)

model = LogisticRegression(C=1.0, max_iter=5000, solver="liblinear", class_weight="balanced", random_state=42)
model.fit(X, y)

feature_names = np.array(tfidf.get_feature_names_out())
coefs = model.coef_[0]

top_ad_idx = np.argsort(coefs)[-20:][::-1]
top_cn_idx = np.argsort(coefs)[:20]

print("\nTop 20 AD-indicative terms:\n")
for term, w in zip(feature_names[top_ad_idx], coefs[top_ad_idx]):
    print(f"{term:25s} {w:.4f}")

print("\nTop 20 CN-indicative terms:\n")
for term, w in zip(feature_names[top_cn_idx], coefs[top_cn_idx]):
    print(f"{term:25s} {w:.4f}")